# Bài 3.3: Tóm tắt Văn bản Trừu tượng (Abstractive Summarization) với `VietAI/vit5`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vietnamese-nlp-mastery/vietnamese-nlp-mastery/blob/main/03_Pretrained_Transformers/03_vit5_abstractive_summarization.ipynb)

> **Mục tiêu bài học:**
> 1. Làm chủ mô hình Encoder-Decoder **`VietAI/vit5-base`** (dựa trên kiến trúc T5 của Google, được tiền huấn luyện trên kho dữ liệu tiếng Việt quy mô lớn).
> 2. Huấn luyện và suy luận (Inference) bài toán Tóm tắt văn bản trên tập dữ liệu **`VietNews`** và đánh giá bằng chỉ số **ROUGE-1 / ROUGE-2 / ROUGE-L**.
> 3. **Lưu ý quan trọng**: Khác với `PhoBERT`, đầu vào của **`ViT5` KHÔNG tách từ bằng dấu `_`** (sử dụng văn bản tiếng Việt tự nhiên đã chuẩn hóa Unicode `NFC`).

In [1]:
import json
from pathlib import Path

vietnews_path = Path("../datasets/samples/vietnews_summarization_sample.jsonl")
with open(vietnews_path, "r", encoding="utf-8") as f:
    articles = [json.loads(line) for line in f]

sample = articles[0]
print(f"Tiêu đề  : {sample['title']}")
print(f"Bài báo  : {sample['article'][:180]}...")
print(f"Tóm tắt  : {sample['abstract']}")

Tiêu đề  : Bước tiến mới của xử lý ngôn ngữ tự nhiên tiếng Việt
Bài báo  : Trong những năm gần đây, lĩnh vực xử lý ngôn ngữ tự nhiên (NLP) tiếng Việt đã chứng kiến sự phát triển mạnh mẽ nhờ nguồn dữ liệu mở từ cộng đồng VLSP, UIT NLP và các viện nghiên...
Tóm tắt  : Các mô hình tiền huấn luyện như PhoBERT, ViT5 và các LLM mã nguồn mở đã giúp cải thiện vượt bậc chất lượng phân tích văn bản tiếng Việt.


## Suy luận & Fine-tuning với `VietAI/vit5-base-vietnews-summarization`

```python
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

model_name = "VietAI/vit5-base-vietnews-summarization"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda")

input_text = f"vietnews: {sample['article']} </s>"
encoding = tokenizer(input_text, return_tensors="pt", max_length=1024, truncation=True).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids=encoding["input_ids"],
        attention_mask=encoding["attention_mask"],
        max_new_tokens=128,
        num_beams=4,
        early_stopping=True,
    )

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Tóm tắt sinh bởi ViT5:", summary)
```